# Unit 3 · How Attention Actually Works
**Learn with Adi - Build an LLM from Scratch**

Executable twin of [Unit 3](https://aditya-402.github.io/learn-with-adi/series/llm-from-scratch/unit3.html).
Each cell mirrors a listing on the page, in order. Run top to bottom and the outputs match the page
(and the reference material, Raschka's *Build a Large Language Model (From Scratch)*) digit for digit.


## The input - six tokens, embedded

In [ ]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     x(1)
   [0.55, 0.87, 0.66], # journey  x(2)
   [0.57, 0.85, 0.64], # starts   x(3)
   [0.22, 0.58, 0.33], # with     x(4)
   [0.77, 0.25, 0.10], # one      x(5)
   [0.05, 0.80, 0.55]] # step     x(6)
)
inputs.shape   # (6 tokens, 3 embedding dims)

## Step 1 - attention scores: dot the query with every token
Expected: `tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])`

In [ ]:
query = inputs[1]                            # x(2), "journey"
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

In [ ]:
# the dot product is just multiply-elementwise-then-sum:
res = 0.
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]
print(res)                          # 0.9544
print(torch.dot(inputs[0], query))  # same

## Step 2 - softmax turns scores into weights
Expected: `tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])`, summing to 1.

In [ ]:
# the obvious fix - divide by the sum (works, but breaks on negatives):
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("plain division:", attn_weights_2_tmp)

# the real fix - softmax:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("softmax:       ", attn_weights_2)
print("sum:", attn_weights_2.sum())

## Step 3 - blend: the context vector z(2)
Expected: `tensor([0.4419, 0.6515, 0.5683])`

In [ ]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

## All six tokens at once - three lines replace all loops

In [ ]:
attn_scores  = inputs @ inputs.T                   # (6,3)@(3,6) -> (6,6)
attn_weights = torch.softmax(attn_scores, dim=-1)  # softmax each ROW
all_context  = attn_weights @ inputs               # (6,6)@(6,3) -> (6,3)
print(all_context)                                 # row 2 == z(2) above

## Trainable weights - the real seed-123 matrices
Expected: `q(2) = tensor([0.4306, 1.4551])`, score = 1.8524, weights `[0.1500, 0.2264, ...]`, z(2) = `[0.3061, 0.8210]`

In [ ]:
x_2  = inputs[1]
d_in = inputs.shape[1]   # 3
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
print(W_query)  # [[0.2961, 0.5166], [0.2517, 0.6886], [0.0740, 0.8665]]

In [ ]:
query_2 = x_2 @ W_query
key_2   = x_2 @ W_key
value_2 = x_2 @ W_value
print("q(2):", query_2)

keys   = inputs @ W_key
values = inputs @ W_value

attn_score_22 = query_2.dot(keys[1])
print("score 22:", attn_score_22)

attn_scores_2 = query_2 @ keys.T
print("all scores:", attn_scores_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print("weights:", attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print("z(2):", context_vec_2)

## SelfAttention_v1 - the mechanism as a class
Expected row 2: `[0.3061, 0.8210]`

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    def forward(self, x):
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

## SelfAttention_v2 - nn.Linear version
Different numbers than v1 (different init), same mechanism. Expected row 1: `[-0.0739, 0.0713]`

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

**Exercise (reference book, ex. 3.1):** transfer sa_v2's weights into a SelfAttention_v1 so both produce identical outputs. Hint: `nn.Linear` stores its weight matrix *transposed*.